In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import math
import glob
from tqdm import tqdm

In [ ]:
VIDEO_FOLDER  = r"C:\AAKASH\MS_NOTES\THESIS\Material\Data\recordings\dataset_refined"
OUTPUT_FOLDER = r"C:\AAKASH\MS_NOTES\THESIS\Material\Data\recordings\extracted_features"
K_NEW_PATH    = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results\newK.npy"
MAP1_PATH     = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results\map1.npy"
MAP2_PATH     = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results\map2.npy"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Physical dimensions in mm
FIELD_WIDTH_MM  = 1200
FIELD_HEIGHT_MM = 680
PAD             = 50
H_PLATFORM      = -118.0
H_ROD           = -85.0
BORDER_WIDTH    = 9.5

# Goal dimensions in mm
GOAL_Y_CENTER   = FIELD_HEIGHT_MM / 2
GOAL_TOLERANCE  = 100

#Color threshold
LOWER_RED = np.array([152, 101, 175], dtype=np.uint8)
UPPER_RED = np.array([175, 255, 255], dtype=np.uint8)

# Filter parameters
BILATERAL_D           = 9
BILATERAL_SIGMA_COLOR = 30
BILATERAL_SIGMA_SPACE = 100
MIN_MOVEMENT_MM       = 3

In [ ]:
# Physical conifiguration of Rods
ROD_CONFIG = {
    1: {'x': 75,   'team': 'Black', 'role': 'GK',  'num_players': 1, 'player_color': 'dark',
        'offsets_bottom': None, 'offsets_top': None},
    2: {'x': 225,  'team': 'Black', 'role': 'DEF', 'num_players': 2, 'player_color': 'dark',
        'offsets_bottom': [15, -219], 'offsets_top': [-15, 219]},
    3: {'x': 375,  'team': 'White', 'role': 'ATK', 'num_players': 3, 'player_color': 'light',
        'offsets_bottom': [-15, -199, -383], 'offsets_top': [15, 199, 383]},
    4: {'x': 525,  'team': 'Black', 'role': 'MID', 'num_players': 5, 'player_color': 'dark',
        'offsets_bottom': [15, -106.5, -228, -349.5, -471], 'offsets_top': [-15, 106.5, 228, 349.5, 471]},
    5: {'x': 675,  'team': 'White', 'role': 'MID', 'num_players': 5, 'player_color': 'light',
        'offsets_bottom': [-15, -136.5, -258, -379.5, -501], 'offsets_top': [15, 136.5, 258, 379.5, 501]},
    6: {'x': 825,  'team': 'Black', 'role': 'ATK', 'num_players': 3, 'player_color': 'dark',
        'offsets_bottom': [15, -169, -353], 'offsets_top': [-15, 169, 353]},
    7: {'x': 975,  'team': 'White', 'role': 'DEF', 'num_players': 2, 'player_color': 'light',
        'offsets_bottom': [-15, -249], 'offsets_top': [15, 249]},
    8: {'x': 1125, 'team': 'White', 'role': 'GK',  'num_players': 1, 'player_color': 'light',
        'offsets_bottom': None, 'offsets_top': None},
}

In [ ]:
# Coordinate system
qr_world_points = np.array([
    [-BORDER_WIDTH, BORDER_WIDTH, H_PLATFORM],
    [FIELD_WIDTH_MM + BORDER_WIDTH, BORDER_WIDTH, H_PLATFORM],
    [FIELD_WIDTH_MM + BORDER_WIDTH, FIELD_HEIGHT_MM - BORDER_WIDTH, H_PLATFORM],
    [-BORDER_WIDTH, FIELD_HEIGHT_MM - BORDER_WIDTH, H_PLATFORM]
], dtype=np.float32)

field_world_points = np.array([
    [0, 0, 0], [FIELD_WIDTH_MM, 0, 0],
    [FIELD_WIDTH_MM, FIELD_HEIGHT_MM, 0], [0, FIELD_HEIGHT_MM, 0]
], dtype=np.float32)

dst_points = np.array([
    [PAD, PAD], [FIELD_WIDTH_MM + PAD, PAD],
    [FIELD_WIDTH_MM + PAD, FIELD_HEIGHT_MM + PAD], [PAD, FIELD_HEIGHT_MM + PAD]
], dtype=np.float32)

In [ ]:
def get_marker_data(img):
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector   = cv2.aruco.ArucoDetector(aruco_dict, cv2.aruco.DetectorParameters())
    corners, ids, _ = detector.detectMarkers(img)
    return corners, ids

def compute_homography_and_pose(img_undist, K_new):
    corners, ids = get_marker_data(img_undist)
    if ids is None:
        return None, None, None
    marker_dict = {int(ids[i][0]): corners[i].reshape(4, 2) for i in range(len(ids))}
    if not all(m in marker_dict for m in [0, 1, 2, 3]):
        return None, None, None
    qr_image_points = np.array([
        marker_dict[0][2], marker_dict[1][3],
        marker_dict[2][0], marker_dict[3][1]
    ], dtype=np.float32)
    success, rvec, tvec = cv2.solvePnP(qr_world_points, qr_image_points, K_new, None, flags=cv2.SOLVEPNP_ITERATIVE)
    if not success:
        return None, None, None
    rvec, tvec = cv2.solvePnPRefineLM(qr_world_points, qr_image_points, K_new, None, rvec, tvec)
    field_corners_img, _ = cv2.projectPoints(field_world_points, rvec, tvec, K_new, None)
    H, _ = cv2.findHomography(field_corners_img.reshape(-1, 2), dst_points)
    return H, rvec, tvec

def project_rod_line_to_warped(rod_x, rvec, tvec, K_new, H, num_samples=200):
    y_positions = np.linspace(0, FIELD_HEIGHT_MM, num_samples)
    rod_3d = np.array([[rod_x, y, H_ROD] for y in y_positions], dtype=np.float32)
    rod_img, _ = cv2.projectPoints(rod_3d, rvec, tvec, K_new, None)
    warped_pts = []
    for pt in rod_img.reshape(-1, 2):
        wpt = H @ np.array([pt[0], pt[1], 1.0])
        warped_pts.append((int(wpt[0]/wpt[2]), int(wpt[1]/wpt[2])))
    return warped_pts, y_positions

def extract_intensities_along_line(image, points):
    h, w = image.shape[:2]
    intensities = []
    for (x, y) in points:
        x, y = max(0, min(w-1, x)), max(0, min(h-1, y))
        if len(image.shape) == 3:
            bgr = image[y, x]
            intensities.append(0.299*bgr[2] + 0.587*bgr[1] + 0.114*bgr[0])
    return np.array(intensities)

def apply_bilateral_filter_1d(intensities):
    signal_2d = intensities.reshape(1, -1).astype(np.float32)
    return cv2.bilateralFilter(signal_2d, BILATERAL_D, BILATERAL_SIGMA_COLOR, BILATERAL_SIGMA_SPACE).flatten()

def find_peaks_above_threshold(inverted, y_positions, threshold):
    is_peak = inverted > threshold
    diff    = np.diff(is_peak.astype(int), prepend=0, append=0)
    starts  = np.where(diff == 1)[0]
    ends    = np.where(diff == -1)[0]
    regions = []
    for s, e in zip(starts, ends):
        s, e = max(0, s), min(len(y_positions)-1, e)
        regions.append({
            'y_start':  y_positions[s],
            'y_end':    y_positions[e],
            'y_center': (y_positions[s]+y_positions[e])/2,
            'width_mm': y_positions[e]-y_positions[s],
            'height':   np.max(inverted[s:e+1]) if e > s else inverted[s]
        })
    return regions

def merge_nearby_regions(regions, merge_gap=10, complete_width=45):
    if not regions:
        return []
    regions.sort(key=lambda r: r['y_start'])
    merged = [regions[0].copy()]
    for r in regions[1:]:
        if (merged[-1]['width_mm'] < complete_width and r['y_start'] - merged[-1]['y_end'] <= merge_gap):
            merged[-1]['y_end']   = r['y_end']
            merged[-1]['height']  = max(merged[-1]['height'], r['height'])
        else:
            merged[-1]['y_center'] = (merged[-1]['y_start']+merged[-1]['y_end'])/2
            merged[-1]['width_mm'] = merged[-1]['y_end']-merged[-1]['y_start']
            merged.append(r.copy())
    merged[-1]['y_center'] = (merged[-1]['y_start']+merged[-1]['y_end'])/2
    merged[-1]['width_mm'] = merged[-1]['y_end']-merged[-1]['y_start']
    return merged

def find_stoppers(regions, player_color):
    req_width = 35 if player_color == 'dark' else 10
    valid = sorted([r for r in regions if r['width_mm'] >= req_width], key=lambda p: p['height'], reverse=True)
    if len(valid) < 2:
        return None, None
    return tuple(sorted(valid[:2], key=lambda p: p['y_center']))

def detect_rod(intensities, y_positions, rod_num, config):
    inverted  = 255 - intensities
    threshold = np.percentile(inverted, 50) + 30
    regions   = find_peaks_above_threshold(inverted, y_positions, threshold)
    merged    = merge_nearby_regions(regions)
    top, bottom = find_stoppers(merged, config['player_color'])
    if not top or not bottom:
        return []
    if config['offsets_bottom'] is None:
        return [(top['y_center']+bottom['y_center'])/2]
    if FIELD_HEIGHT_MM - bottom['y_end'] <= top['y_start']:
        poi, offsets = bottom['y_start'], config['offsets_bottom']
    else:
        poi, offsets = top['y_end'], config['offsets_top']
    return sorted([poi + o for o in offsets])

In [ ]:
def process_video(video_path, K_new, map1, map2):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  ERROR: Cannot open {video_path}")
        return None

    fps          = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dt           = 1.0 / fps if fps > 0 else 1/30.0
    out_w        = int(FIELD_WIDTH_MM + 2 * PAD)
    out_h        = int(FIELD_HEIGHT_MM + 2 * PAD)

    # Kalman filter
    kf = cv2.KalmanFilter(4, 2)
    kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
    kf.transitionMatrix  = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
    kf.processNoiseCov   = np.eye(4, dtype=np.float32) * 0.03

    # Find homography from first valid frame
    H_matrix = rvec_global = tvec_global = None
    for f in range(100):
        ret, frame = cap.read()
        if not ret:
            break
        img_undist = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)
        H_matrix, rvec_global, tvec_global = compute_homography_and_pose(img_undist, K_new)
        if H_matrix is not None:
            break

    if H_matrix is None:
        cap.release()
        print(f"  ERROR: Could not compute homography for {os.path.basename(video_path)}")
        return None

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # Per-video state
    tracking_results  = {}
    hit_frames        = set()
    goal_frames_black = set()
    goal_frames_white = set()

    prev_x = prev_y = None
    prev_v = prev_vx = prev_vy = 0.0
    prev_status   = "Searching"
    goal_counter  = 0
    zone_states   = {}
    last_onfield_x = None
    last_onfield_y = None
    stable_positions = {r: None for r in range(1, 9)}

    def smart_stabilize_local(rod_num, new_pos):
        if len(new_pos) == 0:
            return stable_positions[rod_num] if stable_positions[rod_num] else []
        if stable_positions[rod_num] is None or len(stable_positions[rod_num]) != len(new_pos):
            stable_positions[rod_num] = list(new_pos)
            return new_pos
        if max(abs(new_pos[i] - stable_positions[rod_num][i])
               for i in range(len(new_pos))) <= MIN_MOVEMENT_MM:
            return stable_positions[rod_num]
        stable_positions[rod_num] = list(new_pos)
        return new_pos

    # Frame loop
    frame_num = 0

    with tqdm(total=total_frames, desc=f"  {os.path.basename(video_path)}", leave=False) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            timestamp = frame_num * dt

            # Undistort and warp
            img_undist = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)
            warped     = cv2.warpPerspective(img_undist, H_matrix, (out_w, out_h))

            # Ball detection
            hsv  = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
            mask = cv2.inRange(hsv, LOWER_RED, UPPER_RED)
            mask = cv2.erode(mask,  None, iterations=1)
            mask = cv2.dilate(mask, None, iterations=2)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            measured_pt   = None
            detected_x_mm = detected_y_mm = None
            for cnt in contours:
                ((x, y), radius) = cv2.minEnclosingCircle(cnt)
                if 5 < radius < 35:
                    measured_pt   = np.array([[np.float32(x)], [np.float32(y)]])
                    detected_x_mm = int(x) - PAD
                    detected_y_mm = int(y) - PAD
                    break

            # Kalman predict
            predicted  = kf.predict()
            pred_x_raw = int(predicted[0][0])
            pred_y_raw = int(predicted[1][0])

            # Status logic
            status = "Searching"
            log_x = log_y = None

            if measured_pt is not None:
                kf.correct(measured_pt)

                if 0 <= detected_x_mm <= FIELD_WIDTH_MM and 0 <= detected_y_mm <= FIELD_HEIGHT_MM:
                    # Ball is ON FIELD
                    goal_counter = 0
                    status, log_x, log_y = "On Field", detected_x_mm, detected_y_mm
                    last_onfield_x, last_onfield_y = detected_x_mm, detected_y_mm
                else:
                    # Ball is outside field ("Wall")
                    # Check if near goal area
                    near_left_goal  = detected_x_mm < 0 and abs(detected_y_mm - GOAL_Y_CENTER) < GOAL_TOLERANCE
                    near_right_goal = detected_x_mm > FIELD_WIDTH_MM and abs(detected_y_mm - GOAL_Y_CENTER) < GOAL_TOLERANCE

                    if near_left_goal or near_right_goal:
                        # Ball in goal area - count towards goal
                        goal_counter += 1
                        if goal_counter > 5:
                            status, log_x, log_y = "Goal", "Goal", "Goal"
                        else:
                            status, log_x, log_y = "Wall", "Wall", "Wall"
                    else:
                        # Wall but not near goal - reset
                        goal_counter = 0
                        status, log_x, log_y = "Wall", "Wall", "Wall"
            else:
                # Ball not detected
                pred_x_mm = pred_x_raw - PAD
                pred_y_mm = pred_y_raw - PAD

                # Check if last known position was near goal
                if (last_onfield_x is not None and (last_onfield_x < 50 or last_onfield_x > FIELD_WIDTH_MM - 50) and abs(last_onfield_y - GOAL_Y_CENTER) < GOAL_TOLERANCE):
                    goal_counter += 1
                    if goal_counter > 5:
                        status, log_x, log_y = "Goal", "Goal", "Goal"

                if status != "Goal":
                    if 0 <= pred_x_mm <= FIELD_WIDTH_MM and 0 <= pred_y_mm <= FIELD_HEIGHT_MM:
                        status, log_x, log_y = "Occluded", pred_x_mm, pred_y_mm

            # Goal counting
            if status == "Goal" and prev_status != "Goal":
                if prev_x is not None and isinstance(prev_x, int):
                    if prev_x < 50:
                        goal_frames_white.add(frame_num)
                    elif prev_x > FIELD_WIDTH_MM - 50:
                        goal_frames_black.add(frame_num)
            prev_status = status

            # Ball position for hit detection
            ball_x_mm = ball_y_mm = None
            if isinstance(log_x, int):
                ball_x_mm, ball_y_mm = log_x, log_y

            # Physics
            curr_v = accel = curr_vx = curr_vy = 0.0
            if isinstance(log_x, int) and prev_x is not None and isinstance(prev_x, int):
                dx, dy  = log_x - prev_x, log_y - prev_y
                curr_v  = math.sqrt(dx**2 + dy**2) / 1000.0 / dt
                curr_vx = dx / 1000.0 / dt
                curr_vy = dy / 1000.0 / dt
                accel   = (curr_v - prev_v) / dt

            if isinstance(log_x, int):
                prev_x, prev_y, prev_v = log_x, log_y, curr_v

            # Player detection
            all_player_positions = {}
            for rod_num, config in ROD_CONFIG.items():
                points, y_pos = project_rod_line_to_warped(
                    config['x'], rvec_global, tvec_global, K_new, H_matrix)
                intensities = extract_intensities_along_line(warped, points)
                smoothed    = apply_bilateral_filter_1d(intensities)
                player_pos  = smart_stabilize_local(
                    rod_num, detect_rod(smoothed, y_pos, rod_num, config))
                all_player_positions[rod_num] = player_pos

            prev_vx, prev_vy = curr_vx, curr_vy

            # Logging
            goals_black = len(goal_frames_black)
            goals_white = len(goal_frames_white)

            # Rod positions: 22 individual player Y positions (normalized 0-1)
            expected_players = {1:1, 2:2, 3:3, 4:5, 5:5, 6:3, 7:2, 8:1}
            rod_positions = {}
            for rod_num in range(1, 9):
                positions = all_player_positions.get(rod_num, [])
                for p_idx in range(expected_players[rod_num]):
                    col = f'y_rod_{rod_num}_p{p_idx + 1}'
                    if p_idx < len(positions):
                        rod_positions[col] = round(float(positions[p_idx]) / FIELD_HEIGHT_MM, 4)
                    else:
                        rod_positions[col] = -1

            #4 features computed per frame
            x_ball_norm     = round(float(log_x) / FIELD_WIDTH_MM, 4)  if isinstance(log_x, int) else 0.5
            y_ball_norm     = round(float(log_y) / FIELD_HEIGHT_MM, 4) if isinstance(log_y, int) else 0.5
            ball_detected   = 1 if status in ('On Field', 'Occluded') else 0
            frame_norm      = round(frame_num / 1800, 4)

            tracking_results[frame_num] = {
                # Raw tracking columns
                'Frame':          frame_num,
                'Timestamp':      round(timestamp, 4),
                'X':              log_x,
                'Y':              log_y,
                'Speed_m_s':      round(curr_v, 3),
                'Accel_m_s2':     round(accel, 3),
                'Status':         status,
                'Goals_Black':    goals_black,
                'Goals_White':    goals_white,
                #22 rod player Y positions (normalized 0-1)
                **rod_positions,
                # 4 features
                'x_ball':         x_ball_norm,
                'y_ball':         y_ball_norm,
                'ball_detected':  ball_detected,
                'frame_norm':     frame_norm,
            }
            frame_num += 1
            pbar.update(1)

    cap.release()

    # Save CSV
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    csv_path   = os.path.join(OUTPUT_FOLDER, f"{video_name}_tracking.csv")
    sorted_results = [tracking_results[k] for k in sorted(tracking_results.keys())]
    pd.DataFrame(sorted_results).to_csv(csv_path, index=False)

    goals_black = len(goal_frames_black)
    goals_white = len(goal_frames_white)
    winner = ("Black" if goals_black > goals_white else "White" if goals_white > goals_black else "Draw")

    print(f"  Saved: {video_name}_tracking.csv  |  "
          f"Frames: {frame_num}  |  "
          f"Score Black {goals_black} - White {goals_white}  |  "
          f"Winner: {winner}")

    return csv_path

In [ ]:
K_new = np.load(K_NEW_PATH)
map1  = np.load(MAP1_PATH)
map2  = np.load(MAP2_PATH)
print("Calibration loaded.")

video_files = sorted(glob.glob(os.path.join(VIDEO_FOLDER, "*.mp4")))
if not video_files:
    print(f"No .mp4 files found in {VIDEO_FOLDER}")

print(f"Found {len(video_files)} videos.\n")

success = 0
for i, video_path in enumerate(video_files, 1):
    print(f"[{i}/{len(video_files)}] {os.path.basename(video_path)}")
    result = process_video(video_path, K_new, map1, map2)
    if result:
        success += 1

print(f"\nDone. {success}/{len(video_files)} videos processed successfully.")
print(f"CSVs saved to: {OUTPUT_FOLDER}")